# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import pandas as pd
from IPython.display import display

# Load starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Build feature vector using only pre-prediction information
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "content_type",
    "main_intent"
]

X = df[feature_cols].copy()

# Missingness flags
X["has_word_count"] = X["word_count"].notna().astype(int)
X["has_char_count"] = X["char_count"].notna().astype(int)

# Fill numeric missing values with median
numeric_cols = X.select_dtypes(include="number").columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# Fill categorical missing values explicitly
categorical_cols = X.select_dtypes(exclude="number").columns
X[categorical_cols] = X[categorical_cols].fillna("Unknown")

print("Rows:", X.shape[0])
print("Features:", X.shape[1])

display(X.head())

Rows: 30000
Features: 13


,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,engaged_sessions_90d,content_age_days,days_since_last_update,word_count,char_count,content_type,main_intent,has_word_count,has_char_count
0,3803,29,22,17,1,187,20,3221.0,20457.0,keyword article,transactional,1,1
1,15320,7,10,9,0,445,25,2481.0,15562.0,keyword article,informational,1,1
2,12581,11,14,11,0,141,20,3515.0,23643.0,keyword article,informational,1,1
3,11751,58,87,78,1,463,22,2877.0,19116.0,keyword article,commercial,0,0
4,19140,24,177,145,0,263,14,2803.0,17469.0,keyword article,informational,1,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [4]:
from IPython.display import display, Markdown

feature_notes = pd.DataFrame([
    ["impressions_90d", "Google Search impressions over 90 days", "Median", "Numeric", "YES"],
    ["clicks_90d", "Google Search clicks over 90 days", "Median", "Numeric", "YES"],
    ["pageviews_90d", "Pageviews over 90 days", "Median", "Numeric", "YES"],
    ["sessions_90d", "Sessions over 90 days", "Median", "Numeric", "YES"],
    ["engaged_sessions_90d", "Engaged sessions over 90 days", "Median", "Numeric", "YES"],
    ["content_age_days", "Age of the content", "Median", "Numeric", "YES"],
    ["days_since_last_update", "Days since the content was last updated", "Median", "Numeric", "YES"],
    ["word_count", "Number of words in the content", "Median + missing flag", "Numeric", "YES"],
    ["char_count", "Number of characters in the content", "Median + missing flag", "Numeric", "YES"],
    ["content_type", "Type of content", "Unknown", "Categorical", "YES"],
    ["main_intent", "Main search intent", "Unknown", "Categorical", "YES"],
])

feature_notes.columns = [
    "Feature", "Meaning", "Missing handling",
    "Type", "Available before prediction?"
]

display(feature_notes)

,Feature,Meaning,Missing handling,Type,Available before prediction?
0,impressions_90d,Google Search impressions over 90 days,Median,Numeric,YES
1,clicks_90d,Google Search clicks over 90 days,Median,Numeric,YES
2,pageviews_90d,Pageviews over 90 days,Median,Numeric,YES
3,sessions_90d,Sessions over 90 days,Median,Numeric,YES
4,engaged_sessions_90d,Engaged sessions over 90 days,Median,Numeric,YES
5,content_age_days,Age of the content,Median,Numeric,YES
6,days_since_last_update,Days since the content was last updated,Median,Numeric,YES
7,word_count,Number of words in the content,Median + missing flag,Numeric,YES
8,char_count,Number of characters in the content,Median + missing flag,Numeric,YES
9,content_type,Type of content,Unknown,Categorical,YES


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
from IPython.display import display, Markdown

# Check the original dataset columns for obvious leakage risks
leakage_terms = [
    "target", "label", "future", "refresh", "outcome",
    "updated", "trend", "tier", "score"
]

leakage_candidates = [
    col for col in df.columns
    if any(term in col.lower() for term in leakage_terms)
]

display(Markdown("## Leakage Hunt"))

print("Potential leakage-related columns found:")
print(leakage_candidates)

# Check the feature vector against the candidate list
feature_leakage = [
    col for col in X.columns
    if any(term in col.lower() for term in leakage_terms)
]

print("\nLeakage-related columns included in X:")
print(feature_leakage)

display(Markdown("""
### Leakage test

The feature vector should contain only information available at the moment
the prediction is made.

I checked the source columns for names suggesting labels, future information,
refresh outcomes, or derived scores.

The selected feature vector does not intentionally include a future-window
target or a label-derived field.

**Verdict: PASS**

The features are treated as pre-prediction signals, while any future outcome
used to evaluate the model must remain separate from the feature vector.
"""))

## Leakage Hunt

Potential leakage-related columns found:
['age_tier', 'age_tier_order', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Leakage-related columns included in X:
[]



### Leakage test

The feature vector should contain only information available at the moment
the prediction is made.

I checked the source columns for names suggesting labels, future information,
refresh outcomes, or derived scores.

The selected feature vector does not intentionally include a future-window
target or a label-derived field.

**Verdict: PASS**

The features are treated as pre-prediction signals, while any future outcome
used to evaluate the model must remain separate from the feature vector.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
from IPython.display import display, Markdown

excluded_fields = {
    "content_id": "Identifier only; it does not describe the content's behavior.",
    "client_id": "Identifier for the client; using it could make the model memorize client-specific patterns.",
    "report_date": "Context/date field; using it directly could introduce time-specific patterns rather than content signals.",
    "trend_direction": "Derived from performance behavior and may encode information that is too close to the outcome.",
    "trend_pct": "Derived performance signal that may contain information overlapping with the prediction outcome.",
    "age_tier": "A derived bucket of content age; the raw content_age_days field is more transparent.",
    "freshness_tier": "A derived freshness category; the underlying days_since_last_update field is preferred."
}

display(Markdown("## Fields Excluded"))

for field, reason in excluded_fields.items():
    print(f"- {field}: {reason}")

display(Markdown("""
### Exclusion principle

I excluded identifiers and derived fields that could encourage memorization or
leak information about the outcome. The feature vector should contain signals
that would genuinely be available before the prediction is made.
"""))

## Fields Excluded

- content_id: Identifier only; it does not describe the content's behavior.
- client_id: Identifier for the client; using it could make the model memorize client-specific patterns.
- report_date: Context/date field; using it directly could introduce time-specific patterns rather than content signals.
- trend_direction: Derived from performance behavior and may encode information that is too close to the outcome.
- trend_pct: Derived performance signal that may contain information overlapping with the prediction outcome.
- age_tier: A derived bucket of content age; the raw content_age_days field is more transparent.
- freshness_tier: A derived freshness category; the underlying days_since_last_update field is preferred.



### Exclusion principle

I excluded identifiers and derived fields that could encourage memorization or
leak information about the outcome. The feature vector should contain signals
that would genuinely be available before the prediction is made.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.